# GPT-4o Complete Document Processing Pipeline

This notebook demonstrates a **complete document processing pipeline using GPT-4o** for all three main tasks:

1. **OCR** - Extract text and structured data from document images using GPT-4o Vision
2. **Translation** - Translate documents with text and images side-by-side using GPT-4o VLM
3. **Report Generation** - Generate tables and discrepancy analysis reports

All tasks use **GPT-4o** for consistent, high-quality results with excellent non-Latin script support.

**Note:** This notebook uses GPT-4o only. Any references to other services in older sections can be ignored.

This serves as a guide for integrating these services into the TypeScript/Next.js codebase.


## Setup and Installation

First, install the required Python packages:


In [ ]:
# Install required packages
# Run this in your terminal or uncomment to install:
!pip install openai pillow fpdf2 python-dotenv

## Configuration

Set up your GPT-4o API keys and endpoints. All tasks in this notebook use GPT-4o.


In [ ]:
# ============================================================================
# GPT-4O CONFIGURATION
# ============================================================================


# Initialize GPT-4o client (used throughout the notebook)
from openai import OpenAI
import base64
import json
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
import os
load_dotenv()  # Load environment variables from .env file

GPT4O_ENDPOINT = os.getenv("GPT4O_ENDPOINT")
GPT4O_API_KEY = os.getenv("GPT4O_API_KEY")
GPT4O_DEPLOYMENT = os.getenv("GPT4O_DEPLOYMENT")

client = OpenAI(
    base_url=GPT4O_ENDPOINT,
    api_key=GPT4O_API_KEY
)

## 1. OCR: Extract Text and Structured Data from Documents

Use GPT-4o Vision to extract text and structured data from document images. Returns JSON with text, fields, and tables.


In [3]:
def extract_text_with_gpt4o(image_path):
    """
    Extract text and structured data from a document image using GPT-4o Vision.
    
    Returns JSON structure with:
    - text: All extracted text
    - structured_data: Key-value pairs (fields)
    - tables: Extracted tables
    - document_type: Type of document detected
    - document_language: Detected language
    
    Args:
        image_path: Path to image file
    
    Returns:
        Dictionary with extracted text, structured data, and metadata
    """
    # Convert image to base64
    with open(image_path, 'rb') as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    
    # Determine image MIME type
    image_ext = Path(image_path).suffix.lower()
    mime_type = {
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.png': 'image/png',
        '.gif': 'image/gif',
        '.webp': 'image/webp'
    }.get(image_ext, 'image/jpeg')
    
    prompt = """Extract all text and structured data from this document.

Please provide:
1. All text content in the document, preserving the layout and line breaks where appropriate
2. Identify the document type (e.g., passport, ID card, birth certificate, etc.)
3. Detect the document language
4. Extract key-value pairs for structured documents (e.g., Name, DOB, Document Number, etc.)
5. Extract any tables with headers and rows

Return your response as JSON with this EXACT structure:
{
    "text": "all extracted text here, preserving line breaks",
    "document_type": "type of document (e.g., passport, driver_license, birth_certificate, etc.)",
    "document_language": "language code (e.g., en, fr, ar, zh, etc.)",
    "structured_data": {
        "fields": [
            {"key": "field name", "value": "field value"},
            ...
        ]
    },
    "tables": [
        {
            "headers": ["header1", "header2", ...],
            "rows": [
                ["value1", "value2", ...],
                ...
            ]
        }
    ]
}

Return ONLY valid JSON, no additional text."""
    
    # Build message with image
    content = [
        {
            "type": "text",
            "text": prompt
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{base64_image}"
            }
        }
    ]
    
    messages = [
        {
            "role": "user",
            "content": content
        }
    ]
    
    # Call GPT-4o
    completion = client.chat.completions.create(
        model=GPT4O_DEPLOYMENT,
        messages=messages,
        temperature=0.1,  # Low temperature for accurate OCR
        max_tokens=4000,  # Adjust based on document size
        response_format={"type": "json_object"}  # Force JSON response
    )
    
    response_text = completion.choices[0].message.content
    
    # Parse JSON response
    try:
        parsed = json.loads(response_text)
        return parsed
    except json.JSONDecodeError as e:
        # Fallback: try to extract JSON from response
        import re
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group())
            except:
                pass
        raise Exception(f"Failed to parse JSON response: {e}\nResponse: {response_text}")

# Example usage:
result = extract_text_with_gpt4o("sample_docs\French Haiti\Birth-Certificate-Haiti-handwritten.jpg")
print(json.dumps(result, indent=2, ensure_ascii=False))


{
  "text": "MINISTÈRE DE LA JUSTICE\nÉTAT CIVIL\nLIBERTÉ ÉGALITÉ FRATERNITÉ\nRÉPUBLIQUE D’HAÏTI\nN° ________ B ________\nACTE DE NAISSANCE\nDécl. PÈRE\n2022\n92\n181\nL’an deux mille vingt-deux\net de l’indépendance le jeudi treize octobre à dix heures\nPar-devant Nous, Jean Louis St. Almon\nOfficier de l’état civil de la commune de L’Anse-à-Galets, La Gonâve\nSoussigné : Conformément à l’Arrêté Présidentiel du 13 novembre 2019 accordant à toute personne dépourvue d’acte de naissance, un délai de cinq (5) ans pour faire régulariser son état civil.\nA comparu le sieur ________\ndemeurant et domicilié à Petite Anse\ncommune de L’Anse-à-Galets\nLequel nous a déclaré la naissance de son fils légitime ________\nné à Petite Anse\nle vingt-huit février mil neuf cent soixante-trois à dix heures du matin\n________\ndame née ________\ndemeurant et domiciliée à Petite Anse\n________\n________ enfant il a donné le prénom de Su\nDont acte fait en notre bureau, à L’Anse-à-Galets\nen présence de ___

## 2. Translation: Documents with Text and Images Side-by-Side

Use GPT-4o VLM to translate documents, presenting translated text alongside original images. Perfect for government documents with mixed content.


In [4]:
def translate_document_with_gpt4o(image_path, target_language='en'):
    """
    Translate a document using GPT-4o VLM.
    Presents translated text alongside original images, maintaining document structure.
    
    Args:
        image_path: Path to document image
        target_language: Target language code (e.g., 'en', 'fr', 'ar', 'zh')
    
    Returns:
        Dictionary with original and translated text, structured data, and image references
    """
    # Convert image to base64
    with open(image_path, 'rb') as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    
    # Determine image MIME type
    image_ext = Path(image_path).suffix.lower()
    mime_type = {
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.png': 'image/png',
        '.gif': 'image/gif',
        '.webp': 'image/webp'
    }.get(image_ext, 'image/jpeg')
    
    language_names = {
        'en': 'English',
        'fr': 'French',
        'ar': 'Arabic',
        'zh': 'Chinese',
        'es': 'Spanish',
        'pt': 'Portuguese',
        'de': 'German',
        'ja': 'Japanese',
        'ko': 'Korean',
        'hi': 'Hindi'
    }
    target_lang_name = language_names.get(target_language, target_language)
    
    prompt = f"""Translate this document to {target_lang_name} ({target_language}).

For government documents with text and images:
1. Extract all text from the document
2. Translate all text to {target_lang_name}
3. Preserve the document structure and layout
4. For text found in images (logos, stamps, handwritten text), extract and translate it
5. Present the translation alongside the original, indicating which parts are from images vs regular text

Return your response as JSON with this structure:
{{
    "original_text": "original extracted text",
    "translated_text": "translated text to {target_lang_name}",
    "original_language": "detected language code",
    "target_language": "{target_language}",
    "image_text": {{
        "original": "text found in images/stamps/logos",
        "translated": "translated image text"
    }},
    "structured_data": {{
        "original_fields": [
            {{"key": "field name", "value": "original value"}}
        ],
        "translated_fields": [
            {{"key": "translated field name", "value": "translated value"}}
        ]
    }},
    "layout_preserved": true,
    "notes": "any relevant notes about the translation or document structure"
}}

Return ONLY valid JSON, no additional text."""
    
    # Build message with image
    content = [
        {
            "type": "text",
            "text": prompt
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{base64_image}"
            }
        }
    ]
    
    messages = [
        {
            "role": "user",
            "content": content
        }
    ]
    
    # Call GPT-4o
    completion = client.chat.completions.create(
        model=GPT4O_DEPLOYMENT,
        messages=messages,
        temperature=0.1,  # Low temperature for accurate translation
        max_tokens=4000,
        response_format={"type": "json_object"}
    )
    
    response_text = completion.choices[0].message.content
    
    print("GPT Response: \n", response_text)
    
    # Parse JSON response
    try:
        parsed = json.loads(response_text)
        return parsed
    except json.JSONDecodeError as e:
        import re
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group())
            except:
                pass
        raise Exception(f"Failed to parse JSON response: {e}\nResponse: {response_text}")

# Example usage:
result = translate_document_with_gpt4o("sample_docs\French Haiti\Birth-Certificate-Haiti-handwritten.jpg", target_language='en')
print(json.dumps(result, indent=2, ensure_ascii=False))


GPT Response: 
 {
    "original_text": "MINISTÈRE DE LA JUSTICE\nÉTAT CIVIL\nLIBERTÉ ÉGALITÉ FRATERNITÉ\nRÉPUBLIQUE D’HAÏTI\nACTE DE NAISSANCE\nL’an deux mille vingt-deux\net le jeudi treize octobre à dix heures\nPar-devant Nous, Jean Louis St Alme\nOfficier de l’état civil de la commune de Petite Anse, La Gonave\nSoussigné : Conformément à l’Arrêté Présidentiel du 13 novembre 2019 accordant à toute personne dépourvue d’acte de naissance, un délai de cinq (5) ans pour faire régulariser son état civil.\nA comparu le sieur\ndemeurant et domicilié à Petite Anse\ncommune de l’Anse-à-Galets\nLequel nous a déclaré la naissance de son fils légitime\nné à Petite Anse\nle vingt-huit février mille neuf cent soixante-trois à dix heures du matin\ndame née Tia\ndemeurant et domiciliée à Petite Anse\nauquel enfant il a donné le prénom de Su\nDont acte fait en notre bureau, à l’Anse-à-Galets\nen présence de G\net de M\nmajeurs, demeurant et domiciliés à Petite Anse\ntémoins choisis et amenés par le c

## 3. Report Generation: Immigration Document Analysis

Generate comprehensive reports for legal professionals with:
- **Executive Summary**: Quick overview of findings
- **Personal Information Concordance**: Cross-document comparison of key identifiers (names, DOB, etc.)
- **Document-by-Document Analysis**: Detailed review of each document
- **Cross-Document Discrepancies**: Inconsistencies that need attention
- **Translation Notes**: Accuracy and interpretation concerns
- **Action Items**: Recommended follow-ups for the legal team

In [6]:
def generate_application_report(documents):
    """
    Generate a comprehensive immigration document analysis report.
    
    Designed for lawyers and representatives to:
    - Identify discrepancies across identity documents
    - Flag potential translation issues
    - Cross-reference personal information consistency
    - Provide actionable recommendations
    
    Args:
        documents: List of document dictionaries. Each document should have:
            - name: Document filename
            - extracted_data: Data extracted from OCR (from extract_text_with_gpt4o)
            - translation_data: Translation data (from translate_document_with_gpt4o)
    
    Returns:
        Dictionary with comprehensive report sections
    """
    
    # Prepare documents data for analysis - include both original and translated text
    documents_for_analysis = []
    for doc in documents:
        extracted = doc.get('extracted_data', {})
        translation = doc.get('translation_data', {})
        
        doc_info = {
            'document_name': doc.get('name', 'unknown'),
            'document_type': extracted.get('document_type', 'unknown'),
            'original_language': extracted.get('document_language', 'unknown'),
            'original_text': extracted.get('text', ''),
            'original_fields': extracted.get('structured_data', {}).get('fields', []),
            'translated_text': translation.get('translated_text', ''),
            'translated_fields': translation.get('structured_data', {}).get('translated_fields', []),
            'image_text_original': translation.get('image_text', {}).get('original', ''),
            'image_text_translated': translation.get('image_text', {}).get('translated', ''),
            'translation_notes': translation.get('notes', '')
        }
        documents_for_analysis.append(doc_info)
    
    prompt = f"""You are a legal document analyst specializing in immigration applications. Your task is to analyze identity and official documents for a client/family applying for immigration.

## DOCUMENTS PROVIDED:
{json.dumps(documents_for_analysis, indent=2, ensure_ascii=False)}

## YOUR TASK:
Generate a comprehensive analysis report with the following sections. Be precise, factual, and focus on information relevant to immigration applications.

## REPORT STRUCTURE:

### 1. EXECUTIVE SUMMARY
- Very brief overview (2-3 sentences) of the document set
- Number and types of documents analyzed
- Overall assessment: CLEAR (no issues), MINOR CONCERNS, or REQUIRES ATTENTION
- Key findings summary

### 2. PERSONAL INFORMATION CONCORDANCE TABLE
Create a table comparing key identifying information ACROSS ALL documents:
| Field | Document 1 Value | Document 2 Value | ... | Consistent? |
Focus on: Full Name, Date of Birth, Place of Birth, Parents' Names, Document Numbers, Gender, Nationality, Expiry Dates, etc.

### 3. DOCUMENT-BY-DOCUMENT ANALYSIS
For EACH document, provide:
- Document Type & Issuing Authority
- Issue Date & Validity (if applicable)
- Key Information Extracted (original + translation)
- Document Quality/Legibility Assessment
- Translation Accuracy Notes (if foreign language)
- Flags/Concerns specific to this document

### 4. CROSS-DOCUMENT DISCREPANCIES
List ALL inconsistencies found between documents:
- Name spelling variations (critical for immigration)
- Date discrepancies
- Information conflicts
- Missing information that should be present
For each, indicate: SEVERITY (High/Medium/Low) and EXPLANATION

### 5. TRANSLATION ACCURACY NOTES
- Highlight any terms with multiple possible translations
- Note cultural/legal terms that may need explanation
- Flag any ambiguous or uncertain translations
- Note if handwritten text was difficult to interpret

### 6. ACTION ITEMS
Specific recommendations for the legal team:
- Documents that may need certified re-translation
- Information to verify with the client
- Additional documents to request
- Issues to address in the application

Return your response as JSON with this EXACT structure:
{{
    "executive_summary": {{
        "overview": "brief description of document set",
        "documents_analyzed": 2,
        "document_types": ["type1", "type2"],
        "overall_assessment": "CLEAR | MINOR CONCERNS | REQUIRES ATTENTION",
        "key_findings": ["finding 1", "finding 2"]
    }},
    "personal_info_concordance": {{
        "fields_compared": ["Full Name", "Date of Birth", "Place of Birth", "Parents Names", "Gender", "Nationality"],
        "comparison_table": [
            {{
                "field": "Full Name",
                "values_by_document": [
                    {{"document": "doc1.jpg", "original": "original value", "translated": "translated value"}},
                    {{"document": "doc2.jpg", "original": "original value", "translated": "translated value"}}
                ],
                "is_consistent": true,
                "discrepancy_note": "null or explanation if inconsistent"
            }}
        ],
        "consistency_summary": "summary of overall consistency"
    }},
    "document_analysis": [
        {{
            "document_name": "filename",
            "document_type": "type",
            "issuing_authority": "authority if identifiable",
            "issue_date": "date or N/A",
            "validity": "validity period or N/A",
            "key_information": {{
                "original": {{"field": "value"}},
                "translated": {{"field": "value"}}
            }},
            "legibility_assessment": "Good | Fair | Poor",
            "translation_notes": "notes on translation quality/issues",
            "flags": ["flag 1", "flag 2"]
        }}
    ],
    "cross_document_discrepancies": [
        {{
            "discrepancy_type": "Name Spelling | Date | Information Conflict | Missing Info",
            "description": "detailed description",
            "documents_involved": ["doc1", "doc2"],
            "original_values": ["value1", "value2"],
            "severity": "High | Medium | Low",
            "recommendation": "what to do about it"
        }}
    ],
    "translation_notes": {{
        "ambiguous_terms": [
            {{"term": "original term", "possible_translations": ["option1", "option2"], "context": "explanation"}}
        ],
        "cultural_legal_terms": [
            {{"term": "term", "explanation": "what it means in this context"}}
        ],
        "uncertain_readings": ["description of hard-to-read sections"],
        "overall_translation_confidence": "High | Medium | Low"
    }},
    "action_items": [
        {{
            "priority": "High | Medium | Low",
            "category": "Translation | Verification | Documentation | Application",
            "action": "specific action to take",
            "reason": "why this is needed"
        }}
    ],
    "report_metadata": {{
        "generated_at": "timestamp",
        "total_documents": 2,
        "languages_detected": ["fr", "en"]
    }}
}}

IMPORTANT:
- Compare BOTH original text AND translations when analyzing
- Pay special attention to name spellings - even minor variations matter for immigration
- Note any dates that don't align (birth dates, issue dates, etc.)
- Flag anything that could raise questions during immigration processing
- Be concise but thorough - this report is for legal professionals

Return ONLY valid JSON, no additional text."""
    
    messages = [
        {
            "role": "system",
            "content": "You are an expert immigration document analyst. Generate precise, actionable reports for legal professionals. Focus on factual analysis and flag any inconsistencies that could affect an immigration application."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    
    # Call GPT-4o
    completion = client.chat.completions.create(
        model=GPT4O_DEPLOYMENT,
        messages=messages,
        temperature=0.2,  # Slightly higher for analysis, but still factual
        max_tokens=6000,  # Increased for comprehensive report
        response_format={"type": "json_object"}
    )
    
    response_text = completion.choices[0].message.content
    
    # Parse JSON response
    try:
        parsed = json.loads(response_text)
        return parsed
    except json.JSONDecodeError as e:
        import re
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group())
            except:
                pass
        raise Exception(f"Failed to parse JSON response: {e}\nResponse: {response_text}")


# Example usage - update paths to your local sample documents:
documents = [
    {
        'name': 'birth_certificate.jpg',
        'extracted_data': extract_text_with_gpt4o('sample_docs/French Haiti/Birth-Certificate-Haiti-handwritten.jpg'),
        'translation_data': translate_document_with_gpt4o('sample_docs/French Haiti/Birth-Certificate-Haiti-handwritten.jpg', 'en')
    },
    {
        'name': 'birth_certificate.jpg',
        'extracted_data': extract_text_with_gpt4o('sample_docs\French Haiti\Haitian.birthcertificate handwritten and scanned.png'),
        'translation_data': translate_document_with_gpt4o('sample_docs\French Haiti\Haitian.birthcertificate handwritten and scanned.png', 'en')
    },
    
]
report = generate_application_report(documents)
print(json.dumps(report, indent=4, ensure_ascii=False))

GPT Response: 
 {
    "original_text": "MINISTÈRE DE LA JUSTICE\nÉTAT CIVIL\nLIBERTÉ ÉGALITÉ FRATERNITÉ\nRÉPUBLIQUE D’HAÏTI\nACTE DE NAISSANCE\nL’an deux mille vingt-deux\net de l’indépendance le jeudi treize octobre à dix heures\nPar-devant Nous, Jean Louis St Alme\nOfficier de l’état civil de la commune de L’Anse-à-Galets, La Gonave\nSoussigné : Conformément à l’Arrêté Présidentiel du 13 novembre 2019 accordant à toute personne dépourvue d’acte de naissance, un délai de cinq (5) ans pour faire régulariser son état civil.\nA comparu le sieur\ndemeurant et domicilié à Petite Anse\ncommune de L’Anse-à-Galets\nLequel nous a déclaré la naissance de son fils légitime\nné à Petite Anse\nle vingt-huit février mille neuf cent soixante-trois à dix heures du matin\ndame\ndemeurant et domiciliée à Petite Anse\nauquel enfant il a donné le prénom de Si\nDont acte fait en notre bureau, à L’Anse-à-Galets\nen présence de G\net de M\nmajeurs, demeurant et domiciliés à Petite Anse\ntémoins choisis et a

## 3.1 Save Report as PDF

Export the immigration document analysis report as a well-formatted PDF for legal review and case files.

In [7]:
!pip install fpdf2


   ---------------------------------------- 0/2 [defusedxml]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   ---------------------------------------- 2/2 [fpdf2]



In [9]:
# Install required package for PDF generation
# !pip install fpdf2

from fpdf import FPDF
from datetime import datetime

class ImmigrationReportPDF(FPDF):
    """Custom PDF class for immigration document analysis reports."""
    
    def __init__(self):
        super().__init__()
        self.set_auto_page_break(auto=True, margin=15)
        
    def header(self):
        self.set_font('Helvetica', 'B', 10)
        self.set_text_color(100, 100, 100)
        self.cell(0, 10, 'Immigration Document Analysis Report', align='C')
        self.ln(5)
        self.set_draw_color(200, 200, 200)
        self.line(10, 20, 200, 20)
        self.ln(10)
        
    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, f'Page {self.page_no()}/{{nb}}', align='C')
    
    def section_title(self, title):
        self.set_font('Helvetica', 'B', 14)
        self.set_text_color(0, 51, 102)
        self.cell(0, 10, self._sanitize_text(title), ln=True)
        self.set_draw_color(0, 51, 102)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(5)
        
    def subsection_title(self, title):
        self.set_font('Helvetica', 'B', 11)
        self.set_text_color(51, 51, 51)
        self.cell(0, 8, self._sanitize_text(title), ln=True)
        self.ln(2)
        
    def body_text(self, text):
        self.set_font('Helvetica', '', 10)
        self.set_text_color(0, 0, 0)
        self.multi_cell(0, 6, self._sanitize_text(text))
        self.ln(2)
        
    def bullet_point(self, text, indent=10):
        self.set_font('Helvetica', '', 10)
        self.set_text_color(0, 0, 0)
        self.set_x(indent)
        self.multi_cell(0, 6, f"- {self._sanitize_text(text)}")
        
    def severity_badge(self, severity):
        colors = {
            'High': (220, 53, 69),
            'Medium': (255, 193, 7),
            'Low': (40, 167, 69)
        }
        color = colors.get(severity, (128, 128, 128))
        self.set_fill_color(*color)
        self.set_text_color(255, 255, 255) if severity == 'High' else self.set_text_color(0, 0, 0)
        self.set_font('Helvetica', 'B', 8)
        self.cell(20, 5, severity, fill=True, align='C')
        self.set_text_color(0, 0, 0)
    
    def _sanitize_text(self, text):
        """Remove or replace Unicode characters that Helvetica can't handle."""
        if not isinstance(text, str):
            text = str(text)
        # Replace common Unicode characters with ASCII equivalents
        replacements = {
            '\u2022': '-',    # bullet
            '\u2019': "'",    # right single quote
            '\u2018': "'",    # left single quote
            '\u201c': '"',    # left double quote
            '\u201d': '"',    # right double quote
            '\u2013': '-',    # en dash
            '\u2014': '--',   # em dash
            '\u00e9': 'e',    # e acute (for French)
            '\u00e8': 'e',    # e grave
            '\u00ea': 'e',    # e circumflex
            '\u00e0': 'a',    # a grave
            '\u00e2': 'a',    # a circumflex
            '\u00ee': 'i',    # i circumflex
            '\u00f4': 'o',    # o circumflex
            '\u00fb': 'u',    # u circumflex
            '\u00e7': 'c',    # c cedilla
            '\u00c9': 'E',    # E acute
            '\u00c8': 'E',    # E grave
            '\u00ca': 'E',    # E circumflex
            '\u00c0': 'A',    # A grave
            '\u00c2': 'A',    # A circumflex
            '\u00ce': 'I',    # I circumflex
            '\u00d4': 'O',    # O circumflex
            '\u00db': 'U',    # U circumflex
            '\u00c7': 'C',    # C cedilla
        }
        for unicode_char, ascii_char in replacements.items():
            text = text.replace(unicode_char, ascii_char)
        # Remove any remaining non-latin1 characters
        return text.encode('latin-1', 'replace').decode('latin-1')


def save_report_as_pdf(report, output_path="immigration_report.pdf", client_name="Client"):
    """
    Save the immigration document analysis report as a formatted PDF.
    
    Args:
        report: Dictionary containing the report data (from generate_application_report)
        output_path: Path where the PDF will be saved
        client_name: Name of the client/applicant for the cover page
    
    Returns:
        Path to the saved PDF file
    """
    pdf = ImmigrationReportPDF()
    pdf.alias_nb_pages()
    pdf.add_page()
    
    # Title Page
    pdf.set_font('Helvetica', 'B', 24)
    pdf.set_text_color(0, 51, 102)
    pdf.cell(0, 40, '', ln=True)  # Spacing
    pdf.cell(0, 15, 'IMMIGRATION DOCUMENT', align='C', ln=True)
    pdf.cell(0, 15, 'ANALYSIS REPORT', align='C', ln=True)
    pdf.ln(20)
    
    pdf.set_font('Helvetica', '', 14)
    pdf.set_text_color(51, 51, 51)
    pdf.cell(0, 10, f'Client: {pdf._sanitize_text(client_name)}', align='C', ln=True)
    pdf.cell(0, 10, f'Generated: {datetime.now().strftime("%B %d, %Y at %H:%M")}', align='C', ln=True)
    
    # Executive Summary section
    exec_summary = report.get('executive_summary', {})
    pdf.add_page()
    pdf.section_title('1. EXECUTIVE SUMMARY')
    
    # Overall Assessment with color coding
    assessment = exec_summary.get('overall_assessment', 'N/A')
    assessment_colors = {
        'CLEAR': (40, 167, 69),
        'MINOR CONCERNS': (255, 193, 7),
        'REQUIRES ATTENTION': (220, 53, 69)
    }
    
    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(50, 8, 'Overall Assessment: ')
    color = assessment_colors.get(assessment, (128, 128, 128))
    pdf.set_fill_color(*color)
    pdf.set_text_color(255, 255, 255) if assessment == 'REQUIRES ATTENTION' else pdf.set_text_color(0, 0, 0)
    pdf.cell(50, 8, assessment, fill=True, align='C', ln=True)
    pdf.set_text_color(0, 0, 0)
    pdf.ln(5)
    
    pdf.body_text(f"Documents Analyzed: {exec_summary.get('documents_analyzed', 'N/A')}")
    
    doc_types = exec_summary.get('document_types', [])
    if doc_types:
        pdf.body_text(f"Document Types: {', '.join(doc_types)}")
    
    pdf.body_text(exec_summary.get('overview', ''))
    
    pdf.subsection_title('Key Findings:')
    for finding in exec_summary.get('key_findings', []):
        pdf.bullet_point(finding)
    pdf.ln(5)
    
    # Personal Information Concordance
    concordance = report.get('personal_info_concordance', {})
    pdf.section_title('2. PERSONAL INFORMATION CONCORDANCE')
    
    pdf.body_text(concordance.get('consistency_summary', ''))
    pdf.ln(3)
    
    # Create concordance table
    comparison_table = concordance.get('comparison_table', [])
    if comparison_table:
        pdf.set_font('Helvetica', 'B', 9)
        pdf.set_fill_color(0, 51, 102)
        pdf.set_text_color(255, 255, 255)
        
        col_widths = [35, 55, 55, 45]
        headers = ['Field', 'Document 1', 'Document 2', 'Status']
        for i, header in enumerate(headers):
            pdf.cell(col_widths[i], 8, header, border=1, fill=True, align='C')
        pdf.ln()
        
        pdf.set_font('Helvetica', '', 8)
        pdf.set_text_color(0, 0, 0)
        
        for row in comparison_table:
            field = pdf._sanitize_text(row.get('field', ''))[:20]
            values = row.get('values_by_document', [])
            val1 = pdf._sanitize_text(values[0].get('translated', values[0].get('original', 'N/A')))[:30] if len(values) > 0 else 'N/A'
            val2 = pdf._sanitize_text(values[1].get('translated', values[1].get('original', 'N/A')))[:30] if len(values) > 1 else 'N/A'
            consistent = 'MATCH' if row.get('is_consistent', False) else 'MISMATCH'
            
            fill_color = (144, 238, 144) if consistent == 'MATCH' else (255, 182, 193)
            
            pdf.cell(col_widths[0], 7, field, border=1)
            pdf.cell(col_widths[1], 7, val1, border=1)
            pdf.cell(col_widths[2], 7, val2, border=1)
            pdf.set_fill_color(*fill_color)
            pdf.cell(col_widths[3], 7, consistent, border=1, fill=True, align='C')
            pdf.ln()
    pdf.ln(5)
    
    # Document-by-Document Analysis
    pdf.add_page()
    pdf.section_title('3. DOCUMENT-BY-DOCUMENT ANALYSIS')
    
    for doc in report.get('document_analysis', []):
        pdf.subsection_title(f"Document: {doc.get('document_name', 'Unknown')}")
        
        pdf.set_font('Helvetica', '', 9)
        pdf.cell(50, 6, f"Type: {pdf._sanitize_text(doc.get('document_type', 'N/A'))}")
        pdf.cell(0, 6, f"Issuing Authority: {pdf._sanitize_text(doc.get('issuing_authority', 'N/A'))}", ln=True)
        pdf.cell(50, 6, f"Issue Date: {pdf._sanitize_text(doc.get('issue_date', 'N/A'))}")
        pdf.cell(0, 6, f"Validity: {pdf._sanitize_text(doc.get('validity', 'N/A'))}", ln=True)
        pdf.cell(0, 6, f"Legibility: {pdf._sanitize_text(doc.get('legibility_assessment', 'N/A'))}", ln=True)
        
        if doc.get('translation_notes'):
            pdf.set_font('Helvetica', 'I', 9)
            pdf.multi_cell(0, 5, f"Translation Notes: {pdf._sanitize_text(doc.get('translation_notes'))}")
        
        flags = doc.get('flags', [])
        if flags:
            pdf.set_font('Helvetica', 'B', 9)
            pdf.set_text_color(220, 53, 69)
            pdf.cell(0, 6, "Flags/Concerns:", ln=True)
            pdf.set_text_color(0, 0, 0)
            pdf.set_font('Helvetica', '', 9)
            for flag in flags:
                pdf.bullet_point(flag, indent=15)
        pdf.ln(5)
    
    # Cross-Document Discrepancies
    discrepancies = report.get('cross_document_discrepancies', [])
    if discrepancies:
        pdf.add_page()
        pdf.section_title('4. CROSS-DOCUMENT DISCREPANCIES')
        
        for i, disc in enumerate(discrepancies, 1):
            pdf.set_font('Helvetica', 'B', 10)
            pdf.cell(140, 7, f"Discrepancy #{i}: {pdf._sanitize_text(disc.get('discrepancy_type', 'Unknown'))}")
            pdf.severity_badge(disc.get('severity', 'Medium'))
            pdf.ln(8)
            
            pdf.set_font('Helvetica', '', 9)
            pdf.multi_cell(0, 5, f"Description: {pdf._sanitize_text(disc.get('description', 'N/A'))}")
            
            docs_involved = disc.get('documents_involved', [])
            if docs_involved:
                pdf.cell(0, 5, f"Documents: {', '.join(docs_involved)}", ln=True)
            
            orig_values = disc.get('original_values', [])
            if orig_values:
                pdf.cell(0, 5, f"Values Found: {pdf._sanitize_text(' vs '.join(str(v) for v in orig_values))}", ln=True)
            
            pdf.set_font('Helvetica', 'I', 9)
            pdf.multi_cell(0, 5, f"Recommendation: {pdf._sanitize_text(disc.get('recommendation', 'N/A'))}")
            pdf.ln(5)
    
    # Translation Notes
    translation_notes = report.get('translation_notes', {})
    if translation_notes:
        pdf.add_page()
        pdf.section_title('5. TRANSLATION ACCURACY NOTES')
        
        confidence = translation_notes.get('overall_translation_confidence', 'N/A')
        pdf.body_text(f"Overall Translation Confidence: {confidence}")
        
        ambiguous = translation_notes.get('ambiguous_terms', [])
        if ambiguous:
            pdf.subsection_title('Ambiguous Terms:')
            for term in ambiguous:
                translations = term.get('possible_translations', [])
                pdf.bullet_point(f"{pdf._sanitize_text(term.get('term', ''))}: {', '.join(pdf._sanitize_text(t) for t in translations)}")
                if term.get('context'):
                    pdf.set_x(20)
                    pdf.set_font('Helvetica', 'I', 9)
                    pdf.multi_cell(0, 5, f"Context: {pdf._sanitize_text(term.get('context'))}")
        
        cultural = translation_notes.get('cultural_legal_terms', [])
        if cultural:
            pdf.subsection_title('Cultural/Legal Terms:')
            for term in cultural:
                pdf.bullet_point(f"{pdf._sanitize_text(term.get('term', ''))}: {pdf._sanitize_text(term.get('explanation', ''))}")
        
        uncertain = translation_notes.get('uncertain_readings', [])
        if uncertain:
            pdf.subsection_title('Uncertain/Hard-to-Read Sections:')
            for item in uncertain:
                pdf.bullet_point(pdf._sanitize_text(item))
        pdf.ln(5)
    
    # Action Items
    action_items = report.get('action_items', [])
    if action_items:
        pdf.add_page()
        pdf.section_title('6. ACTION ITEMS')
        
        # Group by priority
        for priority in ['High', 'Medium', 'Low']:
            priority_items = [a for a in action_items if a.get('priority') == priority]
            if priority_items:
                pdf.subsection_title(f'{priority} Priority:')
                for item in priority_items:
                    pdf.set_font('Helvetica', '', 9)
                    category = item.get('category', 'General')
                    action = pdf._sanitize_text(item.get('action', 'N/A'))
                    reason = pdf._sanitize_text(item.get('reason', ''))
                    
                    pdf.bullet_point(f"[{category}] {action}")
                    if reason:
                        pdf.set_x(20)
                        pdf.set_font('Helvetica', 'I', 8)
                        pdf.multi_cell(0, 5, f"Reason: {reason}")
                pdf.ln(3)
    
    # Disclaimer
    pdf.add_page()
    pdf.section_title('DISCLAIMER')
    pdf.set_font('Helvetica', 'I', 9)
    disclaimer_text = """This report is generated using AI-assisted document analysis and is intended to support, not replace, professional legal review. All findings should be verified by qualified legal professionals before use in immigration applications.

The translation and analysis provided are based on automated processing and may not capture all nuances of the original documents. For official purposes, certified translations from accredited translators may be required.

This report is confidential and intended solely for use by the legal team handling this immigration matter."""
    pdf.multi_cell(0, 5, disclaimer_text)
    
    # Save the PDF
    pdf.output(output_path)
    print(f"Report saved to: {output_path}")
    return output_path


# Example usage:
# report = generate_application_report(documents)
save_report_as_pdf(report, "client_immigration_report.pdf", client_name="John Doe")

Report saved to: client_immigration_report.pdf


C:\Users\Admin\AppData\Local\Temp\ipykernel_11048\3977699811.py:125: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 40, '', ln=True)  # Spacing
C:\Users\Admin\AppData\Local\Temp\ipykernel_11048\3977699811.py:126: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 15, 'IMMIGRATION DOCUMENT', align='C', ln=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_11048\3977699811.py:127: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 15, 'ANALYSIS REPORT', align='C', ln=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_11048\3977699811.py:132: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, f'Client: {pdf._sanitize_text(client_

'client_immigration_report.pdf'

## 4. Complete Pipeline Example

Putting it all together: OCR → Translation → Report Generation


In [ ]:
def process_complete_application(document_paths, target_language='en', client_name="Client", output_pdf_path=None):
    """
    Complete pipeline: Process multiple documents through OCR, translation, report generation, and PDF export.
    
    Args:
        document_paths: List of tuples (filename, image_path) for each document
        target_language: Target language for translation (default: 'en')
        client_name: Name of the client/applicant for the report
        output_pdf_path: Path to save PDF report (default: auto-generated)
    
    Returns:
        Complete report with all processed documents and PDF path
    """
    processed_documents = []
    
    print("=" * 70)
    print("IMMIGRATION DOCUMENT ANALYSIS PIPELINE")
    print("=" * 70)
    print(f"Client: {client_name}")
    print(f"Documents to process: {len(document_paths)}")
    print("=" * 70)
    
    for filename, image_path in document_paths:
        print(f"\n📄 Processing: {filename}")
        print("-" * 60)
        
        # Step 1: OCR
        print("1️⃣  Extracting text and structured data...")
        try:
            ocr_result = extract_text_with_gpt4o(image_path)
            print(f"   ✓ Document type: {ocr_result.get('document_type', 'unknown')}")
            print(f"   ✓ Language: {ocr_result.get('document_language', 'unknown')}")
            print(f"   ✓ Extracted {len(ocr_result.get('text', ''))} characters")
            if ocr_result.get('structured_data', {}).get('fields'):
                print(f"   ✓ Found {len(ocr_result['structured_data']['fields'])} structured fields")
        except Exception as e:
            print(f"   ✗ OCR failed: {e}")
            continue
        
        # Step 2: Translation (if needed)
        translation_result = None
        detected_lang = ocr_result.get('document_language', 'unknown')
        if detected_lang != target_language:
            print(f"2️⃣  Translating from {detected_lang} to {target_language}...")
            try:
                translation_result = translate_document_with_gpt4o(image_path, target_language)
                print("   ✓ Translation complete")
            except Exception as e:
                print(f"   ✗ Translation failed: {e}")
        else:
            print(f"2️⃣  No translation needed (already in {target_language})")
        
        # Store processed document
        processed_documents.append({
            'name': filename,
            'extracted_data': ocr_result,
            'translation_data': translation_result
        })
    
    # Step 3: Generate report
    print("\n" + "=" * 70)
    print("GENERATING ANALYSIS REPORT")
    print("=" * 70)
    print("3️⃣  Analyzing documents for discrepancies...")
    
    try:
        report = generate_application_report(processed_documents)
        assessment = report.get('executive_summary', {}).get('overall_assessment', 'N/A')
        print(f"   ✓ Report generated - Assessment: {assessment}")
        
        discrepancies = report.get('cross_document_discrepancies', [])
        if discrepancies:
            print(f"   ⚠ Found {len(discrepancies)} discrepanc{'y' if len(discrepancies) == 1 else 'ies'}")
        else:
            print("   ✓ No cross-document discrepancies found")
            
    except Exception as e:
        print(f"   ✗ Report generation failed: {e}")
        return {
            'documents': processed_documents,
            'error': str(e)
        }
    
    # Step 4: Export to PDF
    print("\n" + "=" * 70)
    print("EXPORTING PDF REPORT")
    print("=" * 70)
    print("4️⃣  Generating PDF document...")
    
    if output_pdf_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_name = "".join(c if c.isalnum() else "_" for c in client_name)
        output_pdf_path = f"immigration_report_{safe_name}_{timestamp}.pdf"
    
    try:
        pdf_path = save_report_as_pdf(report, output_pdf_path, client_name)
        print(f"   ✓ PDF saved: {pdf_path}")
    except Exception as e:
        print(f"   ✗ PDF export failed: {e}")
        pdf_path = None
    
    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE")
    print("=" * 70)
    
    return {
        'documents': processed_documents,
        'report': report,
        'pdf_path': pdf_path
    }


# Example usage:
# document_paths = [
#     ('birth_certificate.jpg', 'sample_docs/French Haiti/Birth-Certificate-Haiti-handwritten.jpg'),
#     ('passport.jpg', 'sample_docs/Other/passport_sample.jpg'),
# ]
# 
# result = process_complete_application(
#     document_paths, 
#     target_language='en',
#     client_name="Marie Jean-Baptiste",
#     output_pdf_path="reports/marie_jean_baptiste_analysis.pdf"
# )
# 
# # Access results
# print("\n📊 Report Summary:")
# print(json.dumps(result['report']['executive_summary'], indent=2, ensure_ascii=False))

## Summary: GPT-4o Immigration Document Analysis Pipeline

### Four Main Functions:

1. **`extract_text_with_gpt4o(image_path)`** - OCR
   - Extracts text, structured data (fields), tables
   - Detects document type and language
   - Returns JSON with all extracted information

2. **`translate_document_with_gpt4o(image_path, target_language)`** - Translation
   - Translates documents with text and images
   - Presents translation alongside original
   - Handles image text (stamps, logos, handwritten)
   - Returns JSON with original and translated content

3. **`generate_application_report(documents)`** - Comprehensive Analysis
   - **Executive Summary**: Overall assessment, key findings
   - **Personal Info Concordance**: Cross-document comparison table
   - **Document Analysis**: Individual review of each document
   - **Discrepancies**: Cross-document inconsistencies with severity ratings
   - **Translation Notes**: Accuracy concerns, ambiguous terms
   - **Action Items**: Prioritized recommendations for legal team

4. **`save_report_as_pdf(report, output_path, client_name)`** - PDF Export
   - Generates professional PDF for legal review
   - Color-coded severity indicators
   - Structured sections for easy navigation
   - Includes disclaimer for legal use

### Report Structure:

| Section | Purpose |
|---------|---------|
| Executive Summary | Quick overview with CLEAR/MINOR CONCERNS/REQUIRES ATTENTION assessment |
| Personal Info Concordance | Table comparing names, DOB, etc. across all documents |
| Document Analysis | Detailed review of each document with flags |
| Cross-Document Discrepancies | All inconsistencies with severity (High/Medium/Low) |
| Translation Notes | Ambiguous terms, cultural context, confidence level |
| Action Items | Prioritized to-do list for the legal team |

### Complete Workflow:

```python
# Process documents and generate PDF report
document_paths = [
    ('birth_cert.jpg', 'path/to/birth_cert.jpg'),
    ('passport.jpg', 'path/to/passport.jpg')
]

result = process_complete_application(
    document_paths, 
    target_language='en',
    client_name="Jane Doe",
    output_pdf_path="jane_doe_analysis.pdf"
)

# Access results
report = result['report']           # Full analysis JSON
pdf_path = result['pdf_path']       # Path to generated PDF
assessment = report['executive_summary']['overall_assessment']
discrepancies = report['cross_document_discrepancies']
action_items = report['action_items']
```

### Key Features for Immigration Legal Work:

- **Name Spelling Variations**: Critical for immigration - flags even minor differences
- **Date Consistency**: Checks birth dates, issue dates across documents
- **Severity Ratings**: High/Medium/Low to prioritize issues
- **Translation Confidence**: Indicates reliability of translations
- **Actionable Items**: Specific tasks categorized by priority

### Next Steps for TypeScript Integration:

1. Use these functions as templates for your API routes
2. Convert Python types to TypeScript interfaces
3. Use the same prompt structure in your TypeScript code
4. Maintain the same JSON response formats for frontend consistency
5. Consider using a PDF generation library like `pdfkit` or `@react-pdf/renderer` for TypeScript